In [1]:
!pip install transformers datasets accelerate evaluate scikit-learn pandas openpyxl -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving dataset.xlsx to dataset.xlsx


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_excel('dataset.xlsx', engine='openpyxl')

df = df[['input_text', 'label']].dropna()

print(f" Total samples: {len(df)}")
print(f" Total labels: {df['label'].nunique()}")
print(f"\n Label distribution:")
print(df['label'].value_counts())

 Total samples: 7424
 Total labels: 30

 Label distribution:
label
SUGGEST_COLD          840
SUGGEST_HOT           648
SUGGEST_FRUIT         372
SUGGEST_SWEET         356
SUGGEST_RELAX         356
SUGGEST_ENERGY        328
SUGGEST_SNACK         298
SUGGEST_COFFEE        276
SUGGEST_HEALTHY       264
SUGGEST_VITAMIN       247
ASK_PRICE             225
SUGGEST_REFRESHING    223
SUGGEST_CAKE          218
SUGGEST_DETOX         215
SUGGEST_FREEZE        207
SUGGEST_CHOCOLATE     199
SUGGEST_TEA           195
SUGGEST_CARAMEL       191
SUGGEST_MATCHA        187
GET_MENU              171
SUGGEST_VEGETABLE     158
SUGGEST_SPICY         157
OTHER                 150
GREETING              150
GET_CATEGORY          150
SUGGEST_LESS_SUGAR    140
THANKS                135
SUGGEST_CRISPY        132
SUGGEST_SALTY         131
SUGGEST_CREAM         105
Name: count, dtype: int64


In [4]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

labels = label_encoder.classes_.tolist()
print(f"\n Label Mapping:")
for i, label in enumerate(labels):
    print(f"  LABEL_{i}: {label}")

NUM_LABELS = len(labels)
print(f"\n Total labels: {NUM_LABELS}")



 Label Mapping:
  LABEL_0: ASK_PRICE
  LABEL_1: GET_CATEGORY
  LABEL_2: GET_MENU
  LABEL_3: GREETING
  LABEL_4: OTHER
  LABEL_5: SUGGEST_CAKE
  LABEL_6: SUGGEST_CARAMEL
  LABEL_7: SUGGEST_CHOCOLATE
  LABEL_8: SUGGEST_COFFEE
  LABEL_9: SUGGEST_COLD
  LABEL_10: SUGGEST_CREAM
  LABEL_11: SUGGEST_CRISPY
  LABEL_12: SUGGEST_DETOX
  LABEL_13: SUGGEST_ENERGY
  LABEL_14: SUGGEST_FREEZE
  LABEL_15: SUGGEST_FRUIT
  LABEL_16: SUGGEST_HEALTHY
  LABEL_17: SUGGEST_HOT
  LABEL_18: SUGGEST_LESS_SUGAR
  LABEL_19: SUGGEST_MATCHA
  LABEL_20: SUGGEST_REFRESHING
  LABEL_21: SUGGEST_RELAX
  LABEL_22: SUGGEST_SALTY
  LABEL_23: SUGGEST_SNACK
  LABEL_24: SUGGEST_SPICY
  LABEL_25: SUGGEST_SWEET
  LABEL_26: SUGGEST_TEA
  LABEL_27: SUGGEST_VEGETABLE
  LABEL_28: SUGGEST_VITAMIN
  LABEL_29: THANKS

 Total labels: 30


In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_encoded'])

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_encoded']),
    y=train_df['label_encoded']
)

print(f'\nClass weights computed for {len(class_weights)} classes.')


Train samples: 5939
Test samples: 1485

Class weights computed for 30 classes.


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

print(f" Loaded {MODEL_NAME} with {NUM_LABELS} labels")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

 Loaded vinai/phobert-base with 30 labels


In [7]:
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(
        examples['input_text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df[['input_text', 'label_encoded']].rename(columns={'label_encoded': 'label'}))
test_dataset = Dataset.from_pandas(test_df[['input_text', 'label_encoded']].rename(columns={'label_encoded': 'label'}))

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f" Train: {len(train_dataset)}, Test: {len(test_dataset)}")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Map:   0%|          | 0/5939 [00:00<?, ? examples/s]

Map:   0%|          | 0/1485 [00:00<?, ? examples/s]

 Train: 5939, Test: 1485


In [8]:
import torch
from torch import nn
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')
precision_metric = evaluate.load('precision')
recall_metric = evaluate.load('recall')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='macro')
    precision = precision_metric.compute(predictions=predictions, references=labels, average='macro')
    recall = recall_metric.compute(predictions=predictions, references=labels, average='macro')
    return {
        'accuracy': acc['accuracy'],
        'f1_macro': f1['f1'],
        'precision_macro': precision['precision'],
        'recall_macro': recall['recall']
    }

training_args = TrainingArguments(
    output_dir='./phobert-generic-classifier',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=10,
    push_to_hub=False,
)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')

        weights = torch.tensor(class_weights, dtype=torch.float32).to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print(' Custom Trainer with Class Weights configured')


 Custom Trainer with Class Weights configured


In [9]:
print(" Starting training...")
trainer.train()

 Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,1.499364,1.176336,0.915825,0.911353,0.919349,0.914220
2,0.396950,0.227338,0.976431,0.978639,0.980393,0.977919
3,0.087864,0.090500,0.984512,0.982265,0.981399,0.983944
4,0.048037,0.069838,0.987205,0.986715,0.986621,0.987382
5,0.016864,0.066337,0.987205,0.985558,0.984842,0.986747
6,0.040755,0.057770,0.987879,0.986305,0.985257,0.987981
7,0.023854,0.055403,0.989899,0.988937,0.987145,0.991065
8,0.008752,0.051331,0.989899,0.989034,0.988349,0.990087
9,0.006783,0.049144,0.991919,0.990665,0.989586,0.992067
10,0.014004,0.048052,0.991919,0.991232,0.990164,0.992561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3720, training_loss=0.3730636374524204, metrics={'train_runtime': 1689.2567, 'train_samples_per_second': 35.157, 'train_steps_per_second': 2.202, 'total_flos': 3907523501706240.0, 'train_loss': 0.3730636374524204, 'epoch': 10.0})

In [10]:
results = trainer.evaluate()
print(f'\n Evaluation Results:')
print(f'  Accuracy: {results["eval_accuracy"]:.2%}')
val1 = results.get("eval_precision_macro", 0)
print(f'  Precision Macro: {val1:.2%}')
val2 = results.get("eval_recall_macro", 0)
print(f'  Recall Macro: {val2:.2%}')
val3 = results.get("eval_f1_macro", 0)
print(f'  F1 Macro: {val3:.2%}')
print(f'  Loss: {results["eval_loss"]:.4f}')



 Evaluation Results:
  Accuracy: 99.19%
  Precision Macro: 99.02%
  Recall Macro: 99.26%
  F1 Macro: 99.12%
  Loss: 0.0481


##  TEST


In [11]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

test_cases = [
    ("xin chào", "GREETING"),
    ("cảm ơn shop", "THANKS"),
    ("menu có gì", "GET_MENU"),
    ("cà phê có món gì", "GET_CATEGORY"),
    ("giá bao nhiêu", "ASK_PRICE"),

    ("có món gì mát không", "SUGGEST_COLD"),
    ("thích uống ngọt", "SUGGEST_SWEET"),
    ("đồ healthy", "SUGGEST_HEALTHY"),
    ("cần tỉnh táo", "SUGGEST_ENERGY"),
    ("ít đường", "SUGGEST_LESS_SUGAR"),
    ("đồ cay", "SUGGEST_SPICY"),
    ("thích ăn mặn", "SUGGEST_SALTY"),
    ("bánh ngọt", "SUGGEST_CAKE"),
    ("uống trà", "SUGGEST_TEA"),
    ("nước ép trái cây", "SUGGEST_FRUIT"),
    ("đá xay", "SUGGEST_FREEZE"),
    ("socola", "SUGGEST_CHOCOLATE"),
    ("detox thanh lọc", "SUGGEST_DETOX"),
]

print("\n" + "="*60)
print(" TEST RESULTS")
print("="*60)

for text, expected in test_cases:
    result = classifier(text)[0]
    label_id = int(result['label'].replace('LABEL_', ''))
    predicted = labels[label_id]
    confidence = result['score']

    status = "oke" if predicted == expected else "error"
    print(f"{status} '{text}'")
    print(f"   Predicted: {predicted} ({confidence:.1%}) | Expected: {expected}")
    print()


 TEST RESULTS
oke 'xin chào'
   Predicted: GREETING (99.6%) | Expected: GREETING

oke 'cảm ơn shop'
   Predicted: THANKS (99.6%) | Expected: THANKS



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


oke 'menu có gì'
   Predicted: GET_MENU (99.6%) | Expected: GET_MENU

oke 'cà phê có món gì'
   Predicted: GET_CATEGORY (99.6%) | Expected: GET_CATEGORY

oke 'giá bao nhiêu'
   Predicted: ASK_PRICE (99.7%) | Expected: ASK_PRICE

oke 'có món gì mát không'
   Predicted: SUGGEST_COLD (99.8%) | Expected: SUGGEST_COLD

oke 'thích uống ngọt'
   Predicted: SUGGEST_SWEET (99.7%) | Expected: SUGGEST_SWEET

oke 'đồ healthy'
   Predicted: SUGGEST_HEALTHY (99.7%) | Expected: SUGGEST_HEALTHY

oke 'cần tỉnh táo'
   Predicted: SUGGEST_ENERGY (99.7%) | Expected: SUGGEST_ENERGY

oke 'ít đường'
   Predicted: SUGGEST_LESS_SUGAR (99.6%) | Expected: SUGGEST_LESS_SUGAR

oke 'đồ cay'
   Predicted: SUGGEST_SPICY (99.7%) | Expected: SUGGEST_SPICY

oke 'thích ăn mặn'
   Predicted: SUGGEST_SALTY (99.6%) | Expected: SUGGEST_SALTY

oke 'bánh ngọt'
   Predicted: SUGGEST_CAKE (99.7%) | Expected: SUGGEST_CAKE

oke 'uống trà'
   Predicted: SUGGEST_TEA (99.7%) | Expected: SUGGEST_TEA

oke 'nước ép trái cây'
   Predicte

## Save

In [12]:
import json

OUTPUT_DIR = "phobert-generic-classifier"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save label mapping
label_mapping = {
    "labels": labels,
    "id2label": {i: label for i, label in enumerate(labels)},
    "label2id": {label: i for i, label in enumerate(labels)}
}

with open(f"{OUTPUT_DIR}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

print(f" Model saved to {OUTPUT_DIR}/")
print(f" Labels: {len(labels)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Model saved to phobert-generic-classifier/
 Labels: 30


In [13]:
import os
import shutil
from google.colab import files

CLEAN_DIR = "phobert_v2"
os.makedirs(CLEAN_DIR, exist_ok=True)

model.save_pretrained(CLEAN_DIR)
tokenizer.save_pretrained(CLEAN_DIR)

shutil.copy("phobert-generic-classifier/label_mapping.json", f"{CLEAN_DIR}/label_mapping.json")

shutil.make_archive(CLEAN_DIR, 'zip', CLEAN_DIR)
print(f" Đã tạo file {CLEAN_DIR}.zip ")

files.download(f"{CLEAN_DIR}.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 Đã tạo file phobert_v2.zip 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>